# 06 - Final Evaluation on the Held-Out Test Sets

**Support Ticket Triage** - Notebook 6 of 6

**This is the only notebook permitted to open the test sets, and it opens them once.**
Every number produced here is final. Nothing is tuned after this point — if a result is
disappointing, it gets reported, not fixed.

### What has been decided on validation already

| Task | Classes | Dummy | TF-IDF + LR | DistilBERT (3 ep) |
|---|---|---|---|---|
| Product | 9 | 0.1103 | **0.7748** | 0.7784 |
| Issue | 15 | 0.0678 | **0.5737** | 0.5729 |

Both differences are inside the noise floor — the issue bootstrap gave a 95% interval of
[-0.009, +0.008] with P(DistilBERT better) = 45.1%.

**Model selection happens here, on validation, before the test set is touched.** Where
several DistilBERT checkpoints exist for a task (a 3-epoch and a 6-epoch run, say), the
one with the higher *validation* macro-F1 is chosen and only that one is evaluated on
test. Choosing on validation is what validation is for; choosing on test would invalidate
the number.

### Settings

- **Accelerator: GPU** (inference only — a few minutes).
- **Internet: ON** for the `transformers` pin.
- **Inputs:** Notebook 03's output, plus every DistilBERT notebook's output
  (04, 05, and 05b if you ran it).

In [ ]:
%pip install -q "transformers<5" "tf-keras" 2>/dev/null
print("installed - if the session restarts, continue from the NEXT cell")

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MUST precede the transformers import

import json, glob, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load as joblib_load

from transformers import (DistilBertTokenizerFast,
                          TFDistilBertForSequenceClassification)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 220)

SEED, MAX_LENGTH, BATCH = 42, 256, 64
WORK, PLOTS = "/kaggle/working", "/kaggle/working/plots"
os.makedirs(PLOTS, exist_ok=True)

print("tensorflow", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

---
## Unseal the test sets

`ALLOW_TEST` is set to `True` here and **nowhere else in the project**. Notebooks 03, 04,
05 and 05b all carry the same guard set to `False`, so any accidental test access in
those notebooks raises rather than silently succeeding.

In [ ]:
ALLOW_TEST = True   # <-- the one place in the project this is true

def find_dir(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if name in dirs:
            return os.path.join(root, name)
    return None

def find_file(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None

SPLITS = find_dir("splits")
assert SPLITS, "splits/ not found - add notebook 03's output."

def load_split(task, name, base=SPLITS):
    if name == "test" and not ALLOW_TEST:
        raise RuntimeError(f"{task}/test is sealed until notebook 06.")
    return pd.read_csv(os.path.join(base, task, f"{name}.csv"))

label_maps = json.load(open(find_file("label_maps.json")))

data = {}
for task in ("product", "issue"):
    data[task] = {s: load_split(task, s) for s in ("train", "val", "test")}
    print(f"{task:8s} train {len(data[task]['train'])} | "
          f"val {len(data[task]['val'])} | TEST {len(data[task]['test'])} "
          f"| {label_maps[task]['num_labels']} classes")

---
## Select checkpoints on validation

Each DistilBERT notebook writes `distilbert_<task>_results.json` next to its
`best_model/`. This collects every candidate, ranks them by the **validation** macro-F1
recorded at training time, and keeps the best per task.

In [ ]:
candidates = {}
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if f.startswith("distilbert_") and f.endswith("_results.json"):
            r = json.load(open(os.path.join(root, f)))
            ck = os.path.join(root, "best_model")
            if os.path.isdir(ck):
                candidates.setdefault(r["task"], []).append(
                    {"val_macro_f1": r["val_macro_f1"], "epochs": r["epochs"],
                     "max_length": r.get("max_length"), "ckpt": ck, "meta": r})

chosen = {}
for task, cands in candidates.items():
    cands.sort(key=lambda c: c["val_macro_f1"], reverse=True)
    print(f"\n=== {task}: {len(cands)} checkpoint(s) ===")
    for i, c in enumerate(cands):
        mark = "  <- selected" if i == 0 else ""
        print(f"  val macro-F1 {c['val_macro_f1']:.4f} | {c['epochs']} epochs | "
              f"{c['ckpt']}{mark}")
    chosen[task] = cands[0]

assert chosen, "No DistilBERT checkpoints found - add notebook 04/05 outputs as inputs."
print("\nselected:", {t: (c["epochs"], c["val_macro_f1"]) for t, c in chosen.items()})

---
## Evaluate

Three models per task on the same held-out rows: a stratified dummy (fit on train), the
TF-IDF pipeline saved by Notebook 03, and the selected DistilBERT checkpoint.

In [ ]:
tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def bert_predict(ckpt, texts, id2label):
    model = TFDistilBertForSequenceClassification.from_pretrained(
        ckpt, use_safetensors=False)
    enc = tok(list(texts), max_length=MAX_LENGTH, truncation=True,
              padding="max_length", return_tensors="np")
    ds = tf.data.Dataset.from_tensor_slices(dict(enc)).batch(BATCH)
    logits = model.predict(ds, verbose=0).logits
    return np.array([id2label[i] for i in logits.argmax(-1)]), logits

rows, preds = [], {}

for task in ("product", "issue"):
    tr, te = data[task]["train"], data[task]["test"]
    y_true = te["label"].values
    classes = label_maps[task]["classes"]
    id2label = {i: c for i, c in enumerate(classes)}

    dummy = DummyClassifier(strategy="stratified", random_state=SEED).fit(
        tr["text"], tr["label"])
    p_dummy = dummy.predict(te["text"])

    pipe = joblib_load(find_file(f"tfidf_lr_{task}.joblib"))
    p_tfidf = pipe.predict(te["text"])

    p_bert, _ = bert_predict(chosen[task]["ckpt"], te["text"], id2label)

    preds[task] = {"y_true": y_true, "dummy": p_dummy,
                   "tfidf": p_tfidf, "bert": p_bert}

    for name, p in (("Stratified dummy", p_dummy),
                    ("TF-IDF + LogReg", p_tfidf),
                    (f"DistilBERT ({chosen[task]['epochs']} ep)", p_bert)):
        rows.append({"task": task, "model": name,
                     "test_macro_f1": round(f1_score(y_true, p, average="macro"), 4),
                     "test_accuracy": round(accuracy_score(y_true, p), 4),
                     "n_test": len(te)})
    print(f"{task} done")

In [ ]:
final = pd.DataFrame(rows)
display(final.pivot(index="model", columns="task",
                    values=["test_macro_f1", "test_accuracy"]))
print()
print(final.to_string(index=False))

### Per-class detail

The macro-F1 headline hides which classes carry it. These reports are what the README's
limitations section should be written from.

In [ ]:
for task in ("product", "issue"):
    d = preds[task]
    print(f"\n{'='*78}\n{task.upper()} - TF-IDF + LogReg (test)\n{'='*78}")
    print(classification_report(d["y_true"], d["tfidf"], zero_division=0))
    print(f"{'='*78}\n{task.upper()} - DistilBERT (test)\n{'='*78}")
    print(classification_report(d["y_true"], d["bert"], zero_division=0))

---
## Is the difference real, on the test set?

The same paired bootstrap used on validation, now on the held-out rows. This is the
number that settles whether the project can claim a difference between the two models.

In [ ]:
boots = {}
for task in ("product", "issue"):
    d = preds[task]
    y, a, b = d["y_true"], d["bert"], d["tfidf"]
    B, rng, n = 1000, np.random.default_rng(SEED), len(y)
    deltas = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        deltas[i] = (f1_score(y[idx], a[idx], average="macro")
                     - f1_score(y[idx], b[idx], average="macro"))
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    boots[task] = {"mean_delta": round(float(deltas.mean()), 4),
                   "ci95": [round(float(lo), 4), round(float(hi), 4)],
                   "p_bert_better": round(float((deltas > 0).mean()), 3),
                   "distinguishable": bool(lo > 0 or hi < 0)}
    verdict = ("DistilBERT ahead" if lo > 0 else
               "TF-IDF ahead" if hi < 0 else "INDISTINGUISHABLE")
    print(f"{task:8s} delta {deltas.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  "
          f"P(bert)={(deltas>0).mean():.1%}  -> {verdict}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 13))
for r, task in enumerate(("product", "issue")):
    d = preds[task]
    classes = label_maps[task]["classes"]
    short = [c[:24] + ("..." if len(c) > 24 else "") for c in classes]
    for c, (name, p) in enumerate((("TF-IDF + LogReg", d["tfidf"]),
                                   ("DistilBERT", d["bert"]))):
        cm = confusion_matrix(d["y_true"], p, labels=classes, normalize="true")
        sns.heatmap(cm, ax=axes[r, c], cmap="Blues", vmin=0, vmax=1, cbar=False,
                    annot=len(classes) <= 10, fmt=".2f",
                    xticklabels=short, yticklabels=short)
        f1 = f1_score(d["y_true"], p, average="macro")
        axes[r, c].set(title=f"{task} - {name} (test macro-F1 {f1:.4f})",
                       xlabel="predicted", ylabel="true")
        plt.setp(axes[r, c].get_xticklabels(), rotation=45, ha="right", fontsize=7)
        plt.setp(axes[r, c].get_yticklabels(), fontsize=7)
plt.tight_layout()
plt.savefig(f"{PLOTS}/08_test_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
plot = final[final["model"] != "Stratified dummy"].copy()
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=final, x="task", y="test_macro_f1", hue="model", ax=ax)
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9)
ax.set(title="Held-out test macro-F1", ylim=(0, 1), ylabel="macro-F1")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{PLOTS}/09_final_results.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Save the final record

In [ ]:
summary = {
    "note": "Test sets opened exactly once, in this notebook. Nothing tuned afterwards.",
    "splits": {t: {s: int(len(data[t][s])) for s in ("train", "val", "test")}
               for t in ("product", "issue")},
    "selected_checkpoints": {
        t: {"epochs": c["epochs"], "val_macro_f1": c["val_macro_f1"],
            "max_length": c["max_length"]}
        for t, c in chosen.items()},
    "test_results": rows,
    "paired_bootstrap_on_test": boots,
    "deployment": ("Streamlit Space serves the TF-IDF pipelines: statistically "
                   "indistinguishable from DistilBERT, CPU-only, no 500MB of weights."),
}
with open(f"{WORK}/final_results.json", "w") as f:
    json.dump(summary, f, indent=2)
final.to_csv(f"{WORK}/final_results.csv", index=False)

print(json.dumps(summary, indent=2))

In [ ]:
# Per-row predictions, for error analysis and the README's example table.
for task in ("product", "issue"):
    d = preds[task]
    out = data[task]["test"][["id", "text"]].copy()
    out["true"] = d["y_true"]
    out["pred_tfidf"] = d["tfidf"]
    out["pred_bert"] = d["bert"]
    out["both_wrong"] = (out.true != out.pred_tfidf) & (out.true != out.pred_bert)
    out["disagree"] = out.pred_tfidf != out.pred_bert
    out.to_csv(f"{WORK}/test_predictions_{task}.csv", index=False)
    print(f"{task}: models disagree on {out.disagree.mean():.1%} of test rows | "
          f"both wrong on {out.both_wrong.mean():.1%}")

print("\nartifacts:", sorted(os.listdir(WORK)))

---
## Next

**Save Version -> Save & Run All.**

That closes the modelling work. What remains:

1. **Streamlit Space** - serves the TF-IDF pipelines (`tfidf_lr_product.joblib`,
   `tfidf_lr_issue.joblib` from Notebook 03's output) plus VADER as a clearly-labelled,
   unvalidated tone indicator. Download those two files from Notebook 03's output pane.
2. **GitHub repo** - notebooks 01-06, the app source, `final_results.json`, and the
   README.

### A note for the README

`test_predictions_*.csv` carries a `disagree` column. The rows where TF-IDF and DistilBERT
disagree, and the rows where **both** are wrong, are the most informative thing in this
project for a limitations section — they show whether the remaining errors are model
failures or label ambiguity. Read twenty of them before writing that section.